### Required Libraries 


In [10]:
import cv2
import numpy as np
import csv
import os

In [11]:
DATASET_DIR = "asl_dataset"
OUTPUT_CSV = "landmarks.csv"

mp_hands = mp.solutions.hands

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)

    header = [f"{i}_{axis}" for i in range(21) for axis in ("x", "y", "z")]
    writer.writerow(header + ["label"])

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:

        for label in sorted(os.listdir(DATASET_DIR)):
            label_dir = os.path.join(DATASET_DIR, label)
            if not os.path.isdir(label_dir):
                continue

            print(f"Processing: {label}")

            for img_name in os.listdir(label_dir):
                img_path = os.path.join(label_dir, img_name)

                img = cv2.imread(img_path)
                if img is None:
                    continue

                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                result = hands.process(img_rgb)

                if not result.multi_hand_landmarks:
                    continue  # skip bad samples

                hand = result.multi_hand_landmarks[0]

                landmarks = []
                for lm in hand.landmark:
                    landmarks.extend([lm.x, lm.y, lm.z])

                writer.writerow(landmarks + [label])

Processing: A


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import joblib

df = pd.read_csv("landmarks.csv")

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

labels = sorted(set(y))
label_map = {l: i for i, l in enumerate(labels)}
inv_label_map = {i: l for l, i in label_map.items()}
y = np.array([label_map[l] for l in y])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    max_iter=500,
    random_state=42
)

model.fit(X_train, y_train)

print("Accuracy:", accuracy_score(y_test, model.predict(X_test)))

joblib.dump(model, "asl_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(inv_label_map, "label_map.pkl")
